In [ ]:
!nvidia-smi | head -12

# GPU benchmark: TokenVector.Inference vs PyTorch trên Colab GPU (T4)

So sánh GPT-2 124M (init ngẫu nhiên, seq1024, fwd+bwd).
**CHỈ GIỮ cách đo công bằng**: cùng VM, cùng input token `[(i*37+11)%50257]`,
cùng seed 7, 2 thread cả hai bên, **xen kẽ A/B 3 vòng mỗi bên, lấy median**.

- Cell `[1/2]`: `.tkv` mode 2 (5 pha) vs torch `tier_e` T=2 — **pha nào thua**
- Cell `[2/2]`: `.tkv` cuBLAS vs torch CUDA — tổng (apples-to-apples GPU)
- Cell `[ram]`: peak RSS `.tkv` vs torch (cùng điều kiện)

> **Chú thích bắt buộc:** loss `.tkv` ≠ torch (init scheme khác, FLOPs giống nhau);
> chỉ so **thời gian**. Đúng đắn chỉ kiểm bằng `tv/tests/...` fixtures/parity.


In [ ]:
# [1/5] Install .NET 10 SDK (full SDK, includes runtime) + required libs
!apt-get update -qq && apt-get install -y -qq build-essential gcc libicu74 libssl3 libkrb5-3 zlib1g >/dev/null 2>&1 || apt-get install -y -qq build-essential gcc libicu-dev libkrb5-dev >/dev/null
!for u in https://dot.net/v1/dotnet-install.sh https://builds.dotnet.microsoft.com/dotnet/scripts/v1/dotnet-install.sh; do curl -fsSL --retry 5 --retry-delay 3 --connect-timeout 20 "$u" -o /tmp/dotnet-install.sh && break; echo "that bai: $u"; done; ls -l /tmp/dotnet-install.sh
!bash /tmp/dotnet-install.sh --channel 10.0 --install-dir /opt/dotnet
import os
os.environ["PATH"] = "/opt/dotnet:" + os.environ["PATH"]
!/opt/dotnet/dotnet --info | head -15


In [ ]:
# [2/5] Clone repos + CUDA runtime libs (cublas/cudart) + LD_LIBRARY_PATH
!git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.git
!git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.Inference.git
!pip install -q nvidia-cublas-cu12 nvidia-cuda-runtime-cu12 2>&1 | tail -2
import os, glob
cublas_dir = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cublas/lib')), None)
cudart_dir = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cuda_runtime/lib')), None)
print('cublas_dir:', cublas_dir)
print('cudart_dir:', cudart_dir)
if cublas_dir:
    os.environ['LD_LIBRARY_PATH'] = cublas_dir + ':' + (cudart_dir or '') + ':' + os.environ.get('LD_LIBRARY_PATH', '')
    print('LD_LIBRARY_PATH =', os.environ['LD_LIBRARY_PATH'])


In [ ]:
# [3/5] Build native .so (C + AVX2 + CUDA dispatch) và kiểm tra CUDA khả dụng
!cd TokenVector/Release/tools/simd_native && gcc -O2 -mavx2 -mfma -shared -fPIC \
  tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c \
  -lm -lpthread -ldl -o tkv_simd_native.dll
!cp TokenVector/Release/tools/simd_native/tkv_simd_native.dll TokenVector.Inference/demo/tkv_simd_native.dll
!cp TokenVector/Release/tools/simd_native/tkv_simd_native.dll TokenVector.Inference/tkv_simd_native.dll
!cd TokenVector/Release/tools/simd_native && gcc -O2 -mavx2 -mfma -shared -fPIC tkv_step124.c -o tkv_step124.dll
!cp TokenVector/Release/tools/simd_native/tkv_step124.dll TokenVector.Inference/demo/tkv_step124.dll
print('Built .so')
import ctypes
lib = ctypes.CDLL('/content/TokenVector.Inference/demo/tkv_simd_native.dll')
lib.TkvCUDA_Available.restype = ctypes.c_int
print('TkvCUDA_Available() =', lib.TkvCUDA_Available(), '(1 = cuBLAS loaded, 0 = CPU fallback)')


In [ ]:
# [3.5/5] Build tkv_torch_fused.so (bind ATen) — cho cac op Linear/SDPA/LN/attention-bwd
# Mac dinh (TKV_TORCH_GPU=0): chay CPU oneDNN (nhu truoc day). TKV_TORCH_GPU=1: chay GPU.
import subprocess, os, sys
torch_dir = subprocess.check_output([sys.executable, '-c', 'import torch,os; print(os.path.dirname(torch.__file__))']).decode().strip()
print('torch:', torch_dir)
inc1 = torch_dir + '/include'
inc2 = torch_dir + '/include/torch/csrc/api/include'
libdir = torch_dir + '/lib'
build_cmd = ('cd /content/TokenVector/Release/tools/simd_native && '
    + 'g++ -O2 -shared -std=c++20 -fPIC torch_fused.cpp -o tkv_torch_fused.so '
    + '-I' + inc1 + ' -I' + inc2 + ' -L' + libdir + ' '
    + '-Wl,-rpath,' + libdir + ' '
    + '-ltorch -ltorch_cpu -lc10 -lc10_cuda -ltorch_cuda -Wl,-rpath-link,' + libdir + ' '
    + '2>&1 | tail -25')
print('--- build ---')
print(subprocess.run(build_cmd, shell=True, capture_output=True, text=True).stdout)
so_path = '/content/TokenVector/Release/tools/simd_native/tkv_torch_fused.so'
print('exists:', os.path.exists(so_path))
os.environ['TKV_TORCH_PATH'] = so_path
os.environ['LD_LIBRARY_PATH'] = torch_dir + '/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
import ctypes
lib = ctypes.CDLL(so_path)
print('ctypes load OK (tkv_torch_fused)')


In [ ]:
# [1/2] Fair apples-to-apples: .tkv mode 2 (5 pha) vs torch tier_e (T=2). Xen ke A/B/A/B x3.
import subprocess, re, statistics as st
DOT = '$(command -v dotnet || echo /opt/dotnet/dotnet)'
def run_cmd(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=1800).stdout
def tkv_phase():
    cmd = 'cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && ' + DOT + ' demo/bseq.dll'
    out = run_cmd(cmd)
    m = re.search(r'fwd=([\d.]+) loss=([\d.]+) bwd=([\d.]+) opt=([\d.]+) copyback=([\d.]+) TONG=([\d.]+)', out)
    keys = ('fwd','ce','bwd','opt','copyback','total')
    return dict(zip(keys, map(float, m.groups())))
def torch_phase():
    cmd = 'cd TokenVector.Inference && TKV_BENCH_DEVICE=cpu TKV_BENCH_THREADS=2 python3 benchmark/bench_torch.py e 1024'
    out = run_cmd(cmd)
    def g(pat):
        m = re.search(pat + r'[^\n]*?median=([\d.]+)', out)
        return float(m.group(1))
    return dict(zero=g('zero_grad'), fwd=g(r'fwd\(\+CE\)'), bwd=g('bwd seq1024'), opt=g('opt.step'), total=g('TOTAL'))
tk = []; tt = []
for r in range(3):
    if r % 2 == 0:   # vong 1 & 3: .tkv chay truoc
        a = tkv_phase(); b = torch_phase()
    else:             # vong 2: torch chay truoc (triet VM drift)
        b = torch_phase(); a = tkv_phase()
    tk.append(a); tt.append(b)
    print('vong %d: .tkv fwd=%.0f CE=%.0f bwd=%.0f opt=%.0f TONG=%.0f | torch fwd=%.0f bwd=%.0f opt=%.0f TONG=%.0f' % (
        r+1, a['fwd'], a['ce'], a['bwd'], a['opt'], a['total'], b['fwd'], b['bwd'], b['opt'], b['total']))
med = lambda xs: st.median(xs)
print('\n==== MEDIAN (n=3, xen ke) ====')
print('             .tkv(BLAS)   torch(T=2)   .tkv/torch')
rows = [('fwd(+CE tham khao)', med([x['fwd']+x['ce'] for x in tk]), med([x['fwd'] for x in tt])),
        ('bwd',             med([x['bwd'] for x in tk]),          med([x['bwd'] for x in tt])),
        ('opt.step',        med([x['opt'] for x in tk]),          med([x['opt'] for x in tt])),
        ('TONG',            med([x['total'] for x in tk]),        med([x['total'] for x in tt]))]
for k, a, b in rows:
    print('%-18s %10.0f %10.0f   %.2fx' % (k, a, b, a/b))


In [ ]:
# [2/2] Fair apples-to-apples GPU: .tkv cuBLAS (TKV_GEMM_CUDA=1) vs PyTorch CUDA. Xen ke x3.
import subprocess, re, statistics as st
DOT = '$(command -v dotnet || echo /opt/dotnet/dotnet)'
def run_cmd(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=1800).stdout
def tkv_cuda():
    cmd = 'cd TokenVector.Inference && printf "1024\n0\n1\n" > tv/tests/_seq_cfg.txt && TKV_GEMM_CUDA=1 ' + DOT + ' demo/bseq.dll'
    out = run_cmd(cmd)
    m = re.search(r'fwd=([\d.]+) ms  bwd=([\d.]+) ms  total=([\d.]+)', out)
    return tuple(float(x) for x in m.groups())
def torch_cuda():
    cmd = 'cd TokenVector.Inference && TKV_BENCH_DEVICE=cuda python3 benchmark/bench_torch.py d 1024'
    out = run_cmd(cmd)
    m = re.search(r'fwd\+bwd seq1024 dev=cuda: median=([\d.]+)', out)
    return float(m.group(1))
print('--- sampler GPU bao phu nhanh cuBLAS ---')
!nohup timeout 400 nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader --loop=2 > /content/gpu2.log 2>&1 & echo sampler_started
tk=[]; tt=[]
for r in range(3):
    if r % 2 == 0:
        a = tkv_cuda(); b = torch_cuda()
    else:
        b = torch_cuda(); a = tkv_cuda()
    tk.append(a[2]); tt.append(b)
    print('vong %d: .tkv cuBLAS total=%.1f ms (fwd=%.0f bwd=%.0f)  torch CUDA=%.1f ms' % (r+1, a[2], a[0], a[1], b))
print('\n==== MEDIAN (n=3, xen ke) ====')
print('.tkv cuBLAS median = %.1f ms | torch CUDA median = %.1f ms | ty le = %.2fx' % (st.median(tk), st.median(tt), st.median(tk)/st.median(tt)))
print('--- GPU trong luc .tkv cuBLAS + torch CUDA chay ---')
!sort -u /content/gpu2.log | head -8


In [ ]:
# [3/2] TACH TUNG OP (diagnostic - KHONG phai benchmark): TKV_PROF=1
# In tong thoi gian theo tung op cho mot buoc fwd+bwd (peak RAM/GEMM khac nhau)
print('--- .tkv MAC DINH (BLAS) + TKV_PROF=1 ---')
!cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && TKV_PROF=1 $(command -v dotnet || echo /opt/dotnet/dotnet) demo/bseq.dll 2>&1 | grep -E 'BPHASE|TKV_PROF'
print('--- .tkv cuBLAS (TKV_GEMM_CUDA=1) + TKV_PROF=1 ---')
!cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && TKV_PROF=1 TKV_GEMM_CUDA=1 $(command -v dotnet || echo /opt/dotnet/dotnet) demo/bseq.dll 2>&1 | grep -E 'BPHASE|TKV_PROF'
print('--- .tkv cuBLAS + ATTN_CUDA (TKV_GEMM_CUDA=1 TKV_ATTN_CUDA=1) + TKV_PROF=1 ---')
!cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && TKV_PROF=1 TKV_GEMM_CUDA=1 TKV_ATTN_CUDA=1 $(command -v dotnet || echo /opt/dotnet/dotnet) demo/bseq.dll 2>&1 | grep -E 'BPHASE|TKV_PROF'


In [ ]:
# [7/7] Peak RAM (VmHWM) cua .tkv vs PyTorch CPU - cung seq1024, mode 0 (fwd+bwd, khong optimizer)
!apt-get install -y -qq time >/dev/null 2>&1 || true
print('--- .tkv MAC DINH (BLAS), mode 0 seq1024, VmPeak/VmHWM ---')
!cd TokenVector.Inference && printf '1024\n0\n1\n' > tv/tests/_seq_cfg.txt && /usr/bin/time -v $(command -v dotnet || echo /opt/dotnet/dotnet) demo/bseq.dll 2>&1 | grep -E 'BSEQ|Maximum resident'
print('--- PyTorch CPU (tier_d = fwd+bwd (T=2), seq1024) ---')
!cd TokenVector.Inference && TKV_BENCH_DEVICE=cpu TKV_BENCH_THREADS=2 /usr/bin/time -v python3 benchmark/bench_torch.py d 1024 2>&1 | grep -E 'TOTAL|Maximum resident'
